# 02 — AlphaFold 3 Setup Documentation & ColabFold Fallback
*Module 05 — `05_structure_prediction/`*

**Status:** Setup documentation only. AF3 official model weights require manual
access approval from Google (form link below). Tonight: document install paths,
run ColabFold fallback if available. The Boltz-2 notebook (01) is the primary
production path.

**状态：** 本 notebook 仅为安装文档。AF3 官方模型权重需向 Google 申请访问许可（见下方链接）。
今晚仅记录安装路径，如可用则运行 ColabFold 备选方案。Boltz-2（notebook 01）为主要生产路径。

---

**Key references:**
- Abramson J. et al. (2024). AlphaFold 3. *Nature* 630:493–500. DOI: 10.1038/s41586-024-07487-w.
- Mirdita M. et al. (2022). ColabFold. *Nature Methods* 19:679–682.
- Evans R. et al. (2021). AlphaFold-Multimer. *bioRxiv* 2021.10.04.463034.


## Install Path A — Official AlphaFold 3 (requires weight access)

```bash
# Step 1: Apply for weight access (may take 1-7 days)
# 步骤1：申请权重访问（可能需要 1-7 天）
# URL: https://docs.google.com/forms/d/e/1FAIpQLSfWZAgo1aYk0O4MuAXZj8sLCt4uemNsnflXLdU5DG_eJ45UFw/viewform

# Step 2: Clone official repo
# 步骤2：克隆官方仓库
git clone https://github.com/google-deepmind/alphafold3.git
cd alphafold3

# Step 3: Install (Docker recommended)
# 步骤3：安装（推荐使用 Docker）
docker pull deepmind/alphafold3
# OR native install:
pip install -e .

# Step 4: Download approved model weights (~1.2 GB)
# 步骤4：下载批准的模型权重
# (Follow instructions in the access approval email)

# Step 5: Run on Laguna (see LAGUNA.md Template B)
# 步骤5：在 Laguna 上运行（见 LAGUNA.md 模板 B）
#   sbatch scripts/slurm/af3_batch.slurm
```

**BLOCKER tonight:** Weight access not yet granted. Track status with PI / Alex.
Use ColabFold (Path B) or Boltz-2 (notebook 01) as fallback.

**今晚阻塞：** 权重访问尚未批准。使用 ColabFold（路径 B）或 Boltz-2（notebook 01）作为备选。


## Install Path B — ColabFold (AF2 + AF3-multimer; no weight approval needed)

ColabFold wraps AlphaFold-Multimer and exposes a local server for structure prediction.
It provides `ipTM` as the interface confidence metric, which serves as a reasonable
affinity proxy for protein–protein interactions.

ColabFold 封装了 AlphaFold-Multimer，提供本地服务器进行结构预测，无需 AF3 权重许可。
ipTM 作为界面置信度指标，可用作蛋白-蛋白相互作用的亲和力代理。

```bash
# Install ColabFold (GPU recommended for speed; CPU feasible for short seqs)
# 安装 ColabFold（GPU 加速推荐，短序列 CPU 可用）
pip install colabfold[alphafold]

# OR on Laguna with module system:
# 在 Laguna 上通过模块系统安装：
module load cuda/12.1
pip install colabfold[alphafold-without-jax]
pip install --upgrade jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Run prediction:
# 运行预测：
colabfold_batch <input.fasta> <output_dir> --model-type AlphaFold2-multimer-v3
```


In [ ]:
# Cell 4 — Check available structure prediction tools
# 检查可用的结构预测工具

import subprocess, sys

tools = {
    "boltz":          ["boltz", "--help"],
    "colabfold_batch": ["colabfold_batch", "--help"],
    "alphafold3":     ["python", "-c", "import alphafold; print(alphafold.__version__)"],
}

available = {}
for tool, cmd in tools.items():
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        available[tool] = (r.returncode == 0)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        available[tool] = False
    print(f"  {tool:20s}: {'✓ available' if available[tool] else '✗ not found'}")

print(f"\nPrimary tool tonight: {'boltz' if available['boltz'] else 'NONE — check install'}")
print("Fallback: colabfold_batch" if available["colabfold_batch"] else "Fallback: not installed")


In [ ]:
# Cell 5 — ColabFold fallback runner (stub if not installed)
# ColabFold 备选运行器（未安装则为存根）

from pathlib import Path
import subprocess, json

def run_af3_or_fallback(
    rbp_seq: str,
    receptor_seq: str,
    rbp_id: str,
    receptor_id: str,
    output_dir: Path,
) -> dict:
    """Run structure prediction using available tool.
    
    Priority: boltz2 > colabfold > stub (NotImplementedError)
    优先顺序: boltz2 > colabfold > 存根（抛出 NotImplementedError）
    
    Returns dict: {pdb_path, ipTM, pLDDT, model_used, success}
    """
    import textwrap, shutil
    
    def _wrap60(seq):
        return "\n".join(textwrap.wrap(seq, 60))
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Check available tools / 检查可用工具
    boltz_ok = shutil.which("boltz") is not None
    colabfold_ok = shutil.which("colabfold_batch") is not None
    
    if boltz_ok:
        # Delegate to boltz (see notebook 01 for full logic)
        # 委托给 boltz（完整逻辑见 notebook 01）
        fasta_path = output_dir / f"{rbp_id}__{receptor_id}.fasta"
        fasta_path.write_text(
            f">A|protein\n{_wrap60(rbp_seq)}\n"
            f">B|protein\n{_wrap60(receptor_seq)}\n"
        )
        result = subprocess.run(
            ["boltz", "predict", str(fasta_path),
             "--out_dir", str(output_dir),
             "--accelerator", "cpu",
             "--recycling_steps", "1",
             "--sampling_steps", "50",
             "--output_format", "pdb"],
            capture_output=True, text=True, timeout=1800
        )
        return {"model_used": "boltz2_2.0.3", "success": result.returncode == 0,
                "pdb_path": str(output_dir), "ipTM": float("nan"), "pLDDT": float("nan")}
    
    elif colabfold_ok:
        # ColabFold path / ColabFold 路径
        fasta_path = output_dir / "input.fasta"
        fasta_path.write_text(f">rbp\n{rbp_seq}\n:receptor\n{receptor_seq}\n")
        result = subprocess.run(
            ["colabfold_batch", str(fasta_path), str(output_dir),
             "--model-type", "AlphaFold2-multimer-v3",
             "--num-recycle", "1"],
            capture_output=True, text=True, timeout=3600
        )
        return {"model_used": "colabfold_af2multimer_v3", "success": result.returncode == 0,
                "pdb_path": str(output_dir), "ipTM": float("nan"), "pLDDT": float("nan")}
    
    else:
        raise NotImplementedError(
            "Neither boltz nor colabfold_batch found. "
            "Install with: pip3 install boltz\n"
            "OR: pip install colabfold[alphafold]\n"
            "For AF3 official: apply at https://forms.gle/... (see Install Path A above)"
        )


## How to Run for Real on Laguna

See `LAGUNA.md` Template B for the full `sbatch` script.

**Quick steps:**
```bash
# 1. SSH to Laguna / SSH 到 Laguna
ssh <username>@laguna.<institution>.edu

# 2. Set up environment / 配置环境
module load cuda/12.1
conda activate igem2026

# 3. Submit AlphaFold 3 batch job / 提交 AF3 批处理作业
sbatch scripts/slurm/af3_batch.slurm

# 4. Or submit Boltz-2 batch (simpler, open source) / 提交 Boltz-2 批处理（更简单，开源）
sbatch scripts/slurm/boltz2_screen.slurm

# 5. Pull results back / 拉取结果
rsync -avz <username>@laguna.<institution>.edu:\$SCRATCH/igem_2026/05_structure_prediction/outputs/ 05_structure_prediction/outputs/
```

For a full batch of all 5 RBP × 4 receptor pairs (20 total), the Boltz-2 GPU run
on a single A100 takes ~4 hours. See `AGENT_REPORT.md` §Laguna Runbook for the
exact `sbatch` invocation and parameter choices.

对于全部 5 RBP × 4 受体对（共20对），Boltz-2 在单 A100 GPU 上约需 4 小时。
见 `AGENT_REPORT.md` §Laguna Runbook 中的完整 `sbatch` 调用和参数选择。
